# Train neural-only IPN 13 nhãn cải tiến trên Google Colab

Notebook này dùng bộ keypoint IPN 13 nhãn trên Google Drive, chọn checkpoint bằng validation macro-F1 và giữ test split ngoài quá trình tuning.

Trước khi chạy: chọn **Runtime → Change runtime type → T4 GPU**. Mặc định notebook đọc `MyDrive/ipn-extract-data`, gồm `train.txt`, `val.txt`, `test.txt` và cache 13 nhãn tại `keypoint_tensor_cache/T64/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, subprocess, sys, time
from collections import deque
from pathlib import Path
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa bật GPU: Runtime > Change runtime type > T4 GPU')
print('GPU:', torch.cuda.get_device_name(0))

## 1. Cấu hình

Notebook tự clone code từ GitHub. Google Drive chỉ cần chứa keypoint cache và các file split; không cần upload source code hay project lên Drive.

In [ ]:
# ===== CHỈNH CÁC GIÁ TRỊ NÀY =====
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
CACHE_ROOT_ON_DRIVE = DRIVE_DATA_ROOT / 'keypoint_tensor_cache'
TRAIN_FILE_ON_DRIVE = DRIVE_DATA_ROOT / 'train.txt'
VAL_FILE_ON_DRIVE = DRIVE_DATA_ROOT / 'val.txt'
TEST_FILE_ON_DRIVE = DRIVE_DATA_ROOT / 'test.txt'
OUTPUT_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'neural_improved_13cls'

REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
GIT_BRANCH = 'HuyenHoang'

COPY_CACHE_TO_COLAB = False  # đọc trực tiếp từ Drive; bật True nếu muốn tăng tốc I/O
NUM_CLASSES = 13
NUM_FRAMES = 64
EPOCHS = 50
BATCH_SIZE = 64  # cấu hình an toàn cho T4; tăng lên 128 nếu còn nhiều VRAM
NUM_WORKERS = 0  # đọc trực tiếp Google Drive ổn định hơn multiprocessing
D_MODEL = 128
LAYERS = 3
HEADS = 4
BACKBONE = 'transformer'  # transformer, tcn, bigru, mlp_mixer
SEEDS = [42, 123, 3407]
RUN_ALL_SEEDS = False  # chạy seed đầu trước; bật True sau khi cấu hình đã ổn

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Chuẩn bị code

In [ ]:
PROJECT_ROOT = Path('/content/dynamic_hand_gesture_recognition')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', GIT_BRANCH], check=True)

NEURAL_DIR = PROJECT_ROOT / 'final' / 'code' / 'neural_only_experiments'
TRAIN_SCRIPT = NEURAL_DIR / 'train_neural_only.py'
TEST_SCRIPT = NEURAL_DIR / 'test_neural_only.py'
for required in (TRAIN_SCRIPT, TEST_SCRIPT):
    if not required.exists():
        raise FileNotFoundError(required)

train_source = TRAIN_SCRIPT.read_text(encoding='utf-8')
if '--select-metric' not in train_source or '--disable-motion-features' not in train_source:
    raise RuntimeError('Branch GitHub chưa có code neural cải tiến.')
print('Project:', PROJECT_ROOT)
print('Commit:', subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

## 3. Kiểm tra và copy dữ liệu

Copy cache vào ổ tạm của Colab giúp DataLoader nhanh hơn. Output/checkpoint vẫn được ghi trực tiếp về Drive.

In [ ]:
for path in (CACHE_ROOT_ON_DRIVE, TRAIN_FILE_ON_DRIVE, VAL_FILE_ON_DRIVE):
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy: {path}')

if COPY_CACHE_TO_COLAB:
    LOCAL_DATA_ROOT = Path('/content/neural_data')
    LOCAL_CACHE_ROOT = LOCAL_DATA_ROOT / 'keypoint_tensor_cache'
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not LOCAL_CACHE_ROOT.exists():
        print('Đang copy cache từ Drive sang Colab...')
        shutil.copytree(CACHE_ROOT_ON_DRIVE, LOCAL_CACHE_ROOT)
    for source in (TRAIN_FILE_ON_DRIVE, VAL_FILE_ON_DRIVE, TEST_FILE_ON_DRIVE):
        if source.exists():
            shutil.copy2(source, LOCAL_DATA_ROOT / source.name)
    CACHE_ROOT = LOCAL_CACHE_ROOT
    TRAIN_FILE = LOCAL_DATA_ROOT / TRAIN_FILE_ON_DRIVE.name
    VAL_FILE = LOCAL_DATA_ROOT / VAL_FILE_ON_DRIVE.name
    TEST_FILE = LOCAL_DATA_ROOT / TEST_FILE_ON_DRIVE.name
else:
    CACHE_ROOT = CACHE_ROOT_ON_DRIVE
    TRAIN_FILE, VAL_FILE, TEST_FILE = TRAIN_FILE_ON_DRIVE, VAL_FILE_ON_DRIVE, TEST_FILE_ON_DRIVE

print('Cache:', CACHE_ROOT)
print('Train:', TRAIN_FILE)
print('Val:', VAL_FILE)
print('Test:', TEST_FILE)

In [ ]:
sys.path.insert(0, str(NEURAL_DIR))
from dataset import NeuralOnlyKeypointDataset
from train_neural_only import class_weights, validate_no_split_overlap

train_set = NeuralOnlyKeypointDataset(TRAIN_FILE, CACHE_ROOT, 'train', NUM_FRAMES)
val_set = NeuralOnlyKeypointDataset(VAL_FILE, CACHE_ROOT, 'val', NUM_FRAMES)
validate_no_split_overlap(train_set.samples, val_set.samples)
if NUM_CLASSES != 13:
    raise ValueError(f'NUM_CLASSES={NUM_CLASSES} đang bị giữ từ phiên cũ. Hãy Restart session rồi Run all để nạp NUM_CLASSES=13.')
train_label_ids = sorted(set(label for _, label in train_set.samples))
val_label_ids = sorted(set(label for _, label in val_set.samples))
expected_label_ids = list(range(NUM_CLASSES))
if train_label_ids != expected_label_ids:
    raise ValueError(f'Train label IDs không khớp: actual={train_label_ids}, expected={expected_label_ids}')
if not set(val_label_ids).issubset(expected_label_ids):
    raise ValueError(f'Validation có label ngoài 0..12: {val_label_ids}')
weights = class_weights(train_set.samples, NUM_CLASSES, 0.5)

print('Train samples:', len(train_set), 'Val samples:', len(val_set))
print('Train classes:', len(train_label_ids), 'IDs:', train_label_ids)
print('Val classes:', len(val_label_ids), 'IDs:', val_label_ids)
print('Tensor shape:', tuple(train_set[0]['keypoints'].shape))
print('Class weight range:', float(weights.min()), '→', float(weights.max()))

## 4. Train

Lần đầu nên để `RUN_ALL_SEEDS=False`. Khi một seed chạy ổn, bật lại để có mean ± std đáng tin cậy.

In [ ]:
def run_train(seed: int) -> Path:
    output_dir = OUTPUT_ROOT / f'{BACKBONE}_seed{seed}'
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, '-u', str(TRAIN_SCRIPT),
        '--backbone', BACKBONE,
        '--cache-root', str(CACHE_ROOT),
        '--train-file', str(TRAIN_FILE),
        '--val-file', str(VAL_FILE),
        '--output-dir', str(output_dir),
        '--num-classes', str(NUM_CLASSES),
        '--num-frames', str(NUM_FRAMES),
        '--batch-size', str(BATCH_SIZE),
        '--epochs', str(EPOCHS),
        '--d-model', str(D_MODEL),
        '--layers', str(LAYERS),
        '--heads', str(HEADS),
        '--dropout', '0.15',
        '--label-smoothing', '0.05',
        '--class-weight-power', '0.5',
        '--jitter-std', '0.005',
        '--frame-mask-probability', '0.05',
        '--keypoint-dropout-probability', '0.03',
        '--select-metric', 'macro_f1',
        '--early-stop-patience', '8',
        '--seed', str(seed),
        '--device', 'cuda:0',
        '--num-workers', str(NUM_WORKERS),
        '--log-every', '20',
    ]
    (output_dir / 'command.json').write_text(json.dumps(command, indent=2), encoding='utf-8')
    print('\nRUN:', ' '.join(command))
    log_path = output_dir / 'train.log'
    tail = deque(maxlen=50)
    with log_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
            tail.append(line.rstrip())
        return_code = process.wait()
    if return_code != 0:
        error_tail = '\n'.join(tail)
        raise RuntimeError(f'Train thất bại (exit={return_code}). Log: {log_path}\n{error_tail}')
    return output_dir

seeds_to_run = SEEDS if RUN_ALL_SEEDS else SEEDS[:1]
trained_dirs = [run_train(seed) for seed in seeds_to_run]
print('Hoàn tất:', trained_dirs)

## 5. Vẽ learning curve

In [ ]:
import matplotlib.pyplot as plt

for output_dir in trained_dirs:
    history = json.loads((output_dir / 'history.json').read_text(encoding='utf-8'))
    epochs = [row['epoch'] for row in history]
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, [row['train']['accuracy'] for row in history], label='train acc')
    plt.plot(epochs, [row['val']['accuracy'] for row in history], label='val acc')
    plt.xlabel('epoch'); plt.legend(); plt.grid(alpha=.3)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, [row['val']['macro_f1'] for row in history], label='val macro-F1')
    plt.xlabel('epoch'); plt.legend(); plt.grid(alpha=.3)
    plt.suptitle(output_dir.name)
    plt.tight_layout(); plt.show()

## 6. Đánh giá test đúng một lần

Chỉ chạy cell này sau khi đã chốt cấu hình và seed bằng validation. Không chỉnh hyperparameter dựa trên test.

In [ ]:
RUN_LOCKED_TEST = False

if RUN_LOCKED_TEST:
    if not TEST_FILE.exists():
        raise FileNotFoundError(TEST_FILE)
    for output_dir in trained_dirs:
        command = [
            sys.executable, '-u', str(TEST_SCRIPT),
            '--checkpoint', str(output_dir / 'best.pt'),
            '--cache-root', str(CACHE_ROOT),
            '--test-file', str(TEST_FILE),
            '--batch-size', str(BATCH_SIZE),
            '--num-workers', str(NUM_WORKERS),
            '--device', 'cuda:0',
            '--output', str(output_dir / 'test_metrics.json'),
        ]
        subprocess.run(command, check=True)
else:
    print('Test đang khóa. Đặt RUN_LOCKED_TEST=True sau khi đã chốt cấu hình bằng validation.')

## Gợi ý xử lý lỗi

- CUDA out of memory: giảm `BATCH_SIZE` từ 64 xuống 32 hoặc 16.
- Train lỗi: xem `train.log` trong output của seed; notebook luôn hiện 50 dòng lỗi cuối.
- Không tìm thấy cache: kiểm tra có thư mục `T64` và cấu trúc class/split bên dưới.
- Báo repository chưa có code mới: kiểm tra `GIT_BRANCH='HuyenHoang'` và chạy lại cell clone.
- Colab bị ngắt: checkpoint `last.pt` vẫn nằm trên Drive, nhưng script hiện tại chưa resume optimizer; chạy lại sẽ train mới và ghi đè output cùng seed.